In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from **https://ollama.com/download** for your operating system:
   - **macOS**: download and install the `.pkg`
   - **Windows**: download and install the `.msi`
   - **Linux**: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and start a local model:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To check that the local server is running, you can test:
   ```bash
   curl http://localhost:11434
   ```

If needed, you can also restart the Ollama server with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [5]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

The context doesn’t include any instructions for running **Olama locally**.

If you meant a different tool from the FAQ, I can help with that.


In [6]:
assistant.search('How do I run Olama locally?')

[{'id': '42c7af7ed5',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Agents',
  'question': 'Run MCP Inspector',
  'answer': 'To run the MCP Inspector, execute the following command in the terminal:\n\n```bash\nnpx @modelcontextprotocol/inspector\n```'},
 {'id': 'e394e6f738',
  'course': 'llm-zoomcamp',
  'section': 'Workshop: Open-Source Data Ingestion (dlt)',
  'question': 'How do I know which tables are in the db?',
  'answer': 'You can use the `db.table_names()` method to list all the tables in the database.'},
 {'id': '9b0e2d6a47',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Project: how do I evaluate a recommender-style RAG (no obvious Q&A ground truth)?',
  'answer': "Two complementary approaches that both score for the evaluation criterion:\n\n1. Synthetic ground truth (same idea as the course, adapted). For each item in your dataset, prompt the LLM with the item's description and ask it to generate ~5 user queries that should return that it

In [19]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output[0]

ResponseOutputMessage(id='msg_0a1c75d8dc5eb8bf006a2f20a074408199a4b182a56142c51d', content=[ResponseOutputText(annotations=[], text='If you’ve just discovered the course, you may still be able to join — it depends on the course’s enrollment status and deadline.\n\nTo know for sure, check:\n- whether registration is still open\n- if there’s a waitlist\n- whether the course allows late enrollment\n- any prerequisites or approval requirements\n\nIf you want, I can help you draft a short message to the course organizer asking if you can still join.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [6]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [20]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

response.output[0]

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration access"}', call_id='call_cuj7spVR9z0HVsxImb653otV', name='search', type='function_call', id='fc_04ff491cf65e02e4006a2f20d50be0819886aa7ad3584b792a', namespace=None, status='completed')

previous: ResponseOutputMessage(id='msg_0c7ef9d414b74a06006a2f201f6c7081988d23fcaa85e9b741', content=[ResponseOutputText(annotations=[], text='Yes—usually you can join if enrollment is still open.\n\nA quick way to check:\n- Look for the course page or syllabus\n- See whether there’s a sign-up/enrollment link\n- Check the deadline or “late enrollment” policy\n- If it’s full or already started, email the instructor or course admin and ask if you can still join\n\nIf you want, I can help you write a short message asking to join the course.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

recent: ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration access"}', call_id='call_cuj7spVR9z0HVsxImb653otV', name='search', type='function_call', id='fc_04ff491cf65e02e4006a2f20d50be0819886aa7ad3584b792a', namespace=None, status='completed')

In [25]:
call = response.output[0]

In [36]:
call.arguments

'{"query":"join course discovered course can I join enrollment late registration access"}'

In [ ]:
import json

args = json.loads(call.arguments) #load the JSON string into a Python dictionary
args

{'query': 'join course discovered course can I join enrollment late registration access'}

In [28]:
call.name

'search'

In [ ]:
import json

call = response.output[0]
args = json.loads(call.arguments) #load the JSON string into a Python dictionary

results = search(**args)
result_json = json.dumps(results, indent=2) #convert the search results to a JSON string with indentation for readability

print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course

In [37]:
type(result_json)

str

In [41]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [30]:
messages.append(call)

In [42]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration access"}', call_id='call_cuj7spVR9z0HVsxImb653otV', name='search', type='function_call', id='fc_04ff491cf65e02e4006a2f20d50be0819886aa7ad3584b792a', namespace=None, status='completed')]

In [43]:
messages.append(function_call_output)

In [44]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration access"}', call_id='call_cuj7spVR9z0HVsxImb653otV', name='search', type='function_call', id='fc_04ff491cf65e02e4006a2f20d50be0819886aa7ad3584b792a', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_cuj7spVR9z0HVsxImb653otV',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and g

In [45]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [46]:
response.output[0]

ResponseOutputMessage(id='msg_04ff491cf65e02e4006a2f2d2deef08198a2c924870dddf0ca', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, the key thing is to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [47]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while submissions are still open.


In [48]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(652, 33)

In [49]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.75   # $0.75 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 4.50  # $4.50 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0006375


---

In [59]:
messages = [
    {'role': 'user', 'content': 'How do I run Olama locally?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

print(response.usage)

response_cost =calculate_gpt54mini_price(response.usage.input_tokens, response.usage.output_tokens)

print("Total Cost: $", round(response_cost["total_cost"], 8))

ResponseUsage(input_tokens=66, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=29, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=95)
Total Cost: $ 0.00018


In [60]:
call = response.output[0]
args = json.loads(call.arguments) #load the JSON string into a Python dictionary

results = search(**args)
result_json = json.dumps(results, indent=2)

function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}


In [61]:
messages.append(call)
messages.append(function_call_output)
messages

[{'role': 'user', 'content': 'How do I run Olama locally?'},
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama locally install start server run models FAQ"}', call_id='call_IzuohVJoAjzNAjFueXYb5YMt', name='search', type='function_call', id='fc_0cb46e67a1949267006a2f3230df9081998bb274d3e2815c4b', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_IzuohVJoAjzNAjFueXYb5YMt',
  'output': '[\n  {\n    "id": "1d0b969028",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: Introduction to LLMs and RAG",\n    "question": "Ollama: How to install Ollama?",\n    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\\n\\n- **macOS**: Download the `.pkg` and install it.\\n- **Windows**: Download the `.msi` and install it.\\n- **Linux**: Run the following command in the terminal:\\n\\n  ```bash\\n  curl -fsSL https://ollama.com/install.sh | sh\\n  

In [62]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

print(response.output_text)

response_cost =calculate_gpt54mini_price(response.usage.input_tokens, response.usage.output_tokens)
print("Total Cost: $", round(response_cost["total_cost"], 8))

If you mean **Ollama** locally, here’s the quick setup:

### 1) Install Ollama
- **macOS**: download and install from [https://ollama.com/download](https://ollama.com/download)
- **Windows**: download the `.msi` installer from the same page
- **Linux**:
```bash
curl -fsSL https://ollama.com/install.sh | sh
```

### 2) Run a model locally
Open a terminal and run:
```bash
ollama run llama3
```

This will:
- download the model,
- start it locally,
- open an interactive chat prompt.

### 3) Check the local server
You can verify Ollama is running with:
```bash
curl http://localhost:11434
```

You should see a response from the Ollama server.

### 4) Use it from Python
Install the Python client:
```bash
pip install ollama
```

Example:
```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": "Hello!"}]
)

print(response['message']['content'])
```

If you want, I can also show you how to run **a different model** or how to use **Ollama wit

---

In [2]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [17]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)


In [18]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course enroll registration can I join" }', call_id='call_ncTaZoHWBEidjiFxJcLOM6wH', name='search', type='function_call', id='fc_042dd1cfd9146ff4006a30631c2918819aaf550dc3168adbdb', namespace=None, status='completed')]

In [12]:
import json

In [19]:
messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True
    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course enroll registration can I join" }


In [20]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course enroll registration can I join" }', call_id='call_ncTaZoHWBEidjiFxJcLOM6wH', name='search', type='function_call', id='fc_042dd1cfd9146ff4006a30631c2918819aaf550dc3168adbdb', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_ncTaZoHWBEidjiFxJcLOM6wH',
  'output': '[\n  {\n    "id": "74eb249bbf",\n   

In [21]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break



iteration #1...
function_call: search {"query":"join course discovered can I join course enrollment register late join FAQ"}
function_call: search {"query":"course signup enrollment late join discovered course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. Also, certificates are only available for the live cohort, not self-paced participation.

If you want, I can also help with how to get started, whether registration is required, or what to do if you joined late.


In [22]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join course enrollment register late join FAQ"}', call_id='call_RRbYuzUsfWNiZZIeFRCHfZZI', name='search', type='function_call', id='fc_07dd4ac161f665cb006a3063844944819a912aec7f1c868e14', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course signup enrollment late join discovered course FAQ"}', call_id=

In [15]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> (str,list):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(last_answer)

        it = it + 1
        if has_function_calls == False:
            break

    return (last_answer, messages)

In [16]:
type(agent_loop)

function

In [32]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama run locally Ollama local installation FAQ"}
iteration #2...
function_call: search {"query":"ollama serve localhost 11434 local server FAQ run llama3 python client"}
iteration #3...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - macOS: download and install from https://ollama.com/download
   - Windows: download the `.msi` from the same page
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model if needed and opens a local chat interface.

3. **Check that the server is running**
   ```bash
   curl http://localhost:11434
   ```
   If it’s working, you should get a response showing the available models.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "He

('To run Ollama locally:\n\n1. **Install Ollama**\n   - macOS: download and install from https://ollama.com/download\n   - Windows: download the `.msi` from the same page\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model if needed and opens a local chat interface.\n\n3. **Check that the server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   If it’s working, you should get a response showing the available models.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you want, I can also show you how to run Ollama in the background with `ollama serve`, or how to connect it from a notebook. Any other

In [25]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered course still join enrollment late join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

One important caveat: if you want to receive a certificate, you need to submit your project while submissions are still being accepted.

If you want, I can also help with other course-related questions like homework, certificates, or the next session dates.


'Yes — you can still join the course.\n\nOne important caveat: if you want to receive a certificate, you need to submit your project while submissions are still being accepted.\n\nIf you want, I can also help with other course-related questions like homework, certificates, or the next session dates.'

In [37]:
agent_loop(instructions, "what is the weather today in Los Angeles?")

iteration #1...
ASSISTANT:
I’m sorry, but I can only help with course-related questions and logistics. Weather is outside the scope of the course FAQ. If you have a question about the course, schedule, assignments, or policies, feel free to ask—and let me know if there are other areas you want to explore.


('I’m sorry, but I can only help with course-related questions and logistics. Weather is outside the scope of the course FAQ. If you have a question about the course, schedule, assignments, or policies, feel free to ask—and let me know if there are other areas you want to explore.',
 [{'role': 'developer',
   'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if th

In [33]:
agent_loop(instructions, "How many states are there in the United States?")

iteration #1...
function_call: search {"query":"How many states are there in the United States"}
iteration #2...
function_call: search {"query":"United States number of states FAQ"}
iteration #3...
ASSISTANT:
There are **50 states** in the United States.

If you want, I can also tell you about the **federal district and territories**.


('There are **50 states** in the United States.\n\nIf you want, I can also tell you about the **federal district and territories**.',
 [{'role': 'developer',
   'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results and then perform more searchers. \n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
  {'role': 'user',
   'content': 'How many states are there in the United States?'},
  ResponseFunctionToolCall(arguments='{"query":"How many states are there in the United States"}', call_id='call_ojBPif7hMgTems8Lq60GRBQj', name='search', type='function_call', id='fc_0f79848d3c6e92fc006a306a03ff4481989cb5fb20d0359db2', namespace=None, status='completed'),
  {'type': 'functi

In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [29]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course enrollment discovered the course can I join"}
iteration #2...
function_call: search {"query":"certificate project submitting while accepting submissions live cohort self-paced peer-review course running"}
iteration #3...
ASSISTANT:
Yes, you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. The course can also be followed in a self-paced way, but certificates are only available if you finish with a live cohort.

If you'd like, I can also explain how certification and project peer review work.


In [30]:
result

"Yes, you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. The course can also be followed in a self-paced way, but certificates are only available if you finish with a live cohort.\n\nIf you'd like, I can also explain how certification and project peer review work."

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result, messages = agent_loop(instructions, question)

In [36]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': "what's queen gambit?"},
 ResponseFunctionToolCall(arguments='{"query":"queen gambit"}', call_id='call_KeLoceg7R9qhVxPSPqUcWfiD', name='search', type='function_call', id='fc_02e1708413a5bcc1006a306a8e

In [38]:
result, messages = agent_loop(instructions, "How many states are there in the United States?")

iteration #1...
function_call: search {"query":"United States states how many states are there"}
iteration #2...
ASSISTANT:
This question is not related to the course or its logistics, so I can’t answer it here.

If you have a course-related question, feel free to ask — and let me know if there are other areas you want to explore.


In [47]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user',
  'content': 'How many states are there in the United States?'},
 ResponseFunctionToolCall(arguments='{"query":"United States states how many states are there"}', call_id='call_IU2Jsy1odb9MDQc9ypCxl1Ms', name='s

In [50]:
search_result =json.loads(messages[-2]["output"])
search_result

[{'id': '31456f4b5f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Are there any lectures/videos? Where are they?',
  'answer': 'Please check the bookmarks and pinned links, especially DataTalks.Club’s YouTube account.'},
 {'id': '69430a79a8',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Vector Search',
  'question': 'What are embeddings?',
  'answer': 'Embeddings refer to the process of converting non-numerical data into numerical data while preserving meaning and context. When similar non-numerical data is input into an embedding algorithm, it should yield similar numerical data. The proximity of these numerical values allows for the use of mathematical semantic similarity algorithms. Related concepts include the "vector space model" and "dimensionality reduction."'},
 {'id': 'e394e6f738',
  'course': 'llm-zoomcamp',
  'section': 'Workshop: Open-Source Data Ingestion (dlt)',
  'question': 'How do I know which tables are in the db?

---

In [1]:
!uv add toyaikit

Resolved 126 packages in 6.17s                                       
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/21.96 KiB           
⠙ Preparing packages... (0/7)-------------------     0 B/21.96 KiB           
docstring-parser     ------------------------------     0 B/21.96 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/72.79 KiB           
docstring-parser     ------------------------------     0 B/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.81 KiB/72.79 KiB         
docstring-parser     ------------------------------ 16.00 KiB/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.81 KiB/72.79 KiB         
docstring-parser     ------------------------------ 16.00 KiB/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 14.81 KiB/72.79 KiB         
docstring-parser 

In [2]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [8]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [14]:
type(search)

function

In [17]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [20]:
search_result = search("when does the course start?")
search_result

[{'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2025.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': 'c774542f32',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Server Error (500) When logging in to course homework using GitHub',
  'answer': 'Additional error text seen:\n\n```\nThird-Party Login Failure\n\nAn error occurred while attempting to login via your third-party account.\n

In [21]:
def search(query: str) -> list[dict[str, str]]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [22]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [23]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [24]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [27]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [29]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [38]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama local course FAQ"}', call_id='ca

In [39]:
result.tokens

TokenUsage(model='gpt-5.4-mini', input_tokens=3158, output_tokens=300)

In [40]:
result.cost

CostInfo(input_cost=Decimal('0.0023685'), output_cost=Decimal('0.00135'), total_cost=Decimal('0.0037185'))

In [41]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [42]:
result2.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama local course FAQ"}', call_id='ca

In [43]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None), EasyInputMessage(content='how to run Olama', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"Olama run Ollama how to run course FAQ"}', cal